# Temperature Downscaling — Data Exploration

**Goal:** Downscale MODIS 1km Land Surface Temperature (LST) to ~5km (coarsened), then use high-resolution predictors (Sentinel-2, LULC, DEM) to reconstruct/downscale back. The original 1km data serves as ground truth for evaluation.

**Study area:** Colorado Front Range, 2022–2023

**Datasets:**
| Source | Native Resolution | Temporal | Role |
|--------|------------------|----------|------|
| MODIS MOD11A1 | 1km | Daily | LST target (coarsen → downscale → evaluate) |
| Sentinel-2 | 10m | ~2/month | NDVI, NDWI predictors |
| NLCD LULC | 30m | Annual (static) | Land cover predictor |
| ASTER GDEM | 30m | Static | Elevation predictor |

## 0. Setup

In [ ]:
!pip install -q rasterio geopandas pyhdf rioxarray xarray matplotlib huggingface_hub

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
import geopandas as gpd
from pyhdf.SD import SD, SDC
from shapely.geometry import box
import xarray as xr
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

## 1. Download Data from Hugging Face

Download the dataset from [`akhot2/downscaling`](https://huggingface.co/datasets/akhot2/downscaling) on Hugging Face.

**Note:** The full dataset is ~37 GB. For exploration we download only a subset of MODIS and Sentinel-2 files (January 2022, one tile each). ASTER, LULC, and bounding box are small and downloaded in full. Remove the `allow_patterns` / `ignore_patterns` to get everything.

In [ ]:
from huggingface_hub import snapshot_download

DATA_ROOT = 'data'

# Download small/static folders in full, skip MODIS and Sentinel-2
snapshot_download(
    repo_id='akhot2/downscaling',
    repo_type='dataset',
    local_dir=DATA_ROOT,
    ignore_patterns=['MODIS/*', 'Sentinel2/*'],
)

# Download a MODIS subset: ~first 3 weeks of Jan 2022 (DOY 001–019), all tiles
snapshot_download(
    repo_id='akhot2/downscaling',
    repo_type='dataset',
    local_dir=DATA_ROOT,
    allow_patterns=['MODIS/*.A2022001.*', 'MODIS/*.A2022002.*', 'MODIS/*.A2022003.*',
                    'MODIS/*.A2022004.*', 'MODIS/*.A2022005.*', 'MODIS/*.A2022006.*',
                    'MODIS/*.A2022007.*', 'MODIS/*.A2022008.*', 'MODIS/*.A2022009.*',
                    'MODIS/*.A2022010.*', 'MODIS/*.A2022011.*', 'MODIS/*.A2022012.*',
                    'MODIS/*.A2022013.*', 'MODIS/*.A2022014.*', 'MODIS/*.A2022015.*',
                    'MODIS/*.A2022016.*', 'MODIS/*.A2022017.*', 'MODIS/*.A2022018.*',
                    'MODIS/*.A2022019.*', 'MODIS/*.A2022020.*', 'MODIS/*.A2022021.*'],
)

# Download a Sentinel-2 sample: first available date, tile T13SDD (3 bands)
snapshot_download(
    repo_id='akhot2/downscaling',
    repo_type='dataset',
    local_dir=DATA_ROOT,
    allow_patterns=['Sentinel2/*20220128*T13SDD*.tif'],
)

# Set paths
ASTER_DIR = os.path.join(DATA_ROOT, 'ASTER')
BBOX_DIR  = os.path.join(DATA_ROOT, 'bounding box')
LULC_DIR  = os.path.join(DATA_ROOT, 'LULC')
MODIS_DIR = os.path.join(DATA_ROOT, 'MODIS')
SEN2_DIR  = os.path.join(DATA_ROOT, 'Sentinel2')

# Verify
print()
for name, path in [('ASTER', ASTER_DIR), ('Bounding Box', BBOX_DIR),
                    ('LULC', LULC_DIR), ('MODIS', MODIS_DIR), ('Sentinel2', SEN2_DIR)]:
    exists = os.path.isdir(path)
    count = len(os.listdir(path)) if exists else 0
    print(f"{'OK' if exists else '!!'} {name:14s} — {count} files")

## 2. Study Area — Bounding Box

Load the Colorado Front Range shapefile and visualize the study domain.

In [ ]:
study_area = gpd.read_file(os.path.join(BBOX_DIR, 'colorado_front_range_domain.shp'))

print(f"CRS:    {study_area.crs}")
print(f"Bounds: {study_area.total_bounds}")
print(f"Columns: {list(study_area.columns)}")
print(study_area)

fig, ax = plt.subplots(figsize=(8, 8))
study_area.boundary.plot(ax=ax, color='red', linewidth=2)
ax.set_title('Colorado Front Range — Study Domain')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Store bounds for clipping later
bbox_geom = study_area.geometry.values
minx, miny, maxx, maxy = study_area.total_bounds
print(f"\nStudy area extent: lon [{minx:.3f}, {maxx:.3f}], lat [{miny:.3f}, {maxy:.3f}]")

## 3. ASTER DEM (30m, Static)

The ASTER Global DEM v3 provides elevation data at ~30m resolution. Four tiles cover our study area (N39–N40, W105–W106). We'll mosaic them and clip to the study domain.

In [ ]:
# List all ASTER DEM files (skip the *_num.tif quality files)
dem_files = sorted(glob.glob(os.path.join(ASTER_DIR, '*_dem.tif')))
print(f"DEM tiles: {len(dem_files)}")

for f in dem_files:
    with rasterio.open(f) as src:
        data = src.read(1)
        print(f"  {os.path.basename(f)}: {src.height}x{src.width}, "
              f"res={src.res}, CRS={src.crs}, "
              f"elevation range=[{data.min()}, {data.max()}]m")

In [ ]:
# Mosaic all DEM tiles into a single raster
src_files = [rasterio.open(f) for f in dem_files]
dem_mosaic, dem_transform = merge(src_files)
dem_crs = src_files[0].crs
for s in src_files:
    s.close()

dem_mosaic = dem_mosaic[0]  # single band
print(f"Mosaic shape: {dem_mosaic.shape}")
print(f"Elevation range: {dem_mosaic.min()}m to {dem_mosaic.max()}m")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full mosaic
im0 = axes[0].imshow(dem_mosaic, cmap='terrain',
                       extent=[dem_transform[2],
                               dem_transform[2] + dem_transform[0] * dem_mosaic.shape[1],
                               dem_transform[5] + dem_transform[4] * dem_mosaic.shape[0],
                               dem_transform[5]])
axes[0].set_title('ASTER DEM — Full Mosaic')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
plt.colorbar(im0, ax=axes[0], label='Elevation (m)', shrink=0.8)

# Plot with study area overlay
im1 = axes[1].imshow(dem_mosaic, cmap='terrain',
                       extent=[dem_transform[2],
                               dem_transform[2] + dem_transform[0] * dem_mosaic.shape[1],
                               dem_transform[5] + dem_transform[4] * dem_mosaic.shape[0],
                               dem_transform[5]])
study_area.boundary.plot(ax=axes[1], color='red', linewidth=2)
axes[1].set_title('ASTER DEM — with Study Area')
axes[1].set_xlabel('Longitude')
plt.colorbar(im1, ax=axes[1], label='Elevation (m)', shrink=0.8)

plt.tight_layout()
plt.show()

# Elevation histogram
fig, ax = plt.subplots(figsize=(10, 4))
valid = dem_mosaic[dem_mosaic > -100]
ax.hist(valid.ravel(), bins=100, color='saddlebrown', alpha=0.7, edgecolor='black', linewidth=0.3)
ax.set_xlabel('Elevation (m)')
ax.set_ylabel('Pixel count')
ax.set_title('ASTER DEM — Elevation Distribution')
ax.axvline(valid.mean(), color='red', linestyle='--', label=f'Mean: {valid.mean():.0f}m')
ax.legend()
plt.tight_layout()
plt.show()

## 4. LULC — NLCD Land Cover (30m, Annual)

National Land Cover Database provides land use/land cover classifications at 30m. We have annual maps for 2021–2024.

In [ ]:
# NLCD class legend
NLCD_CLASSES = {
    11: ('Open Water', '#476BA0'),
    12: ('Perennial Ice/Snow', '#D1DDF9'),
    21: ('Developed, Open Space', '#DDC9C9'),
    22: ('Developed, Low Intensity', '#D89382'),
    23: ('Developed, Medium Intensity', '#ED0000'),
    24: ('Developed, High Intensity', '#AA0000'),
    31: ('Barren Land', '#B2ADA3'),
    41: ('Deciduous Forest', '#68AB63'),
    42: ('Evergreen Forest', '#1C6330'),
    43: ('Mixed Forest', '#B5C98E'),
    51: ('Dwarf Scrub', '#A58C30'),
    52: ('Shrub/Scrub', '#CCBA7C'),
    71: ('Grassland/Herbaceous', '#E2E2C1'),
    72: ('Sedge/Herbaceous', '#C9C977'),
    73: ('Lichens', '#99C147'),
    74: ('Moss', '#77AD93'),
    81: ('Pasture/Hay', '#DBD83D'),
    82: ('Cultivated Crops', '#AA7028'),
    90: ('Woody Wetlands', '#BAD8EA'),
    95: ('Emergent Herbaceous Wetlands', '#70A3BA'),
}

lulc_files = sorted(glob.glob(os.path.join(LULC_DIR, 'Annual_NLCD_LndCov_*.tiff')))
print(f"LULC files: {len(lulc_files)}")
for f in lulc_files:
    year = os.path.basename(f).split('_')[3]
    with rasterio.open(f) as src:
        print(f"  {year}: {src.height}x{src.width}, res={src.res}, CRS={src.crs}")

In [ ]:
# Visualize LULC for 2022 (our study period)
lulc_2022 = [f for f in lulc_files if '2022' in os.path.basename(f)][0]

with rasterio.open(lulc_2022) as src:
    lulc_data = src.read(1)
    lulc_transform = src.transform
    lulc_crs = src.crs
    lulc_bounds = src.bounds

print(f"LULC 2022 shape: {lulc_data.shape}")
print(f"Unique classes: {np.unique(lulc_data)}")

# Count pixels per class
unique, counts = np.unique(lulc_data, return_counts=True)
print(f"\nClass distribution:")
for cls, cnt in sorted(zip(unique, counts), key=lambda x: -x[1]):
    name = NLCD_CLASSES.get(cls, ('Unknown', '#000000'))[0]
    pct = 100 * cnt / counts.sum()
    if pct > 0.5:
        print(f"  {cls:3d} {name:35s} {pct:6.1f}% ({cnt:,} pixels)")

In [ ]:
# Plot LULC map with proper NLCD colors
from matplotlib.colors import ListedColormap, BoundaryNorm

# Build colormap from NLCD classes present in data
classes_present = sorted([c for c in np.unique(lulc_data) if c in NLCD_CLASSES])
colors = [NLCD_CLASSES[c][1] for c in classes_present]
cmap = ListedColormap(colors)
bounds = classes_present + [classes_present[-1] + 1]
norm = BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(14, 10))
extent = [lulc_bounds.left, lulc_bounds.right, lulc_bounds.bottom, lulc_bounds.top]
im = ax.imshow(lulc_data, cmap=cmap, norm=norm, extent=extent, interpolation='nearest')
ax.set_title('NLCD Land Cover 2022 — Full Extent')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')

# Add legend
patches = [mpatches.Patch(color=NLCD_CLASSES[c][1], label=f"{c} {NLCD_CLASSES[c][0]}")
           for c in classes_present]
ax.legend(handles=patches, loc='upper left', fontsize=7, ncol=2,
          framealpha=0.9, bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

## 5. MODIS MOD11A1 — Land Surface Temperature (1km, Daily)

MOD11A1 provides daily daytime and nighttime LST at 1km resolution in HDF-EOS format. Each file covers one MODIS sinusoidal tile. Our study area spans tiles **h09v04**, **h09v05**, and **h10v04**.

Key SDS layers:
- `LST_Day_1km` — Daytime LST (scale factor 0.02, units: Kelvin)
- `LST_Night_1km` — Nighttime LST
- `QC_Day` / `QC_Night` — Quality control flags
- `Emis_31` / `Emis_32` — Emissivity bands

In [ ]:
# Inventory MODIS HDF files
modis_hdf = sorted(glob.glob(os.path.join(MODIS_DIR, 'MOD11A1.*.hdf')))
print(f"Total MODIS HDF files: {len(modis_hdf)}")

# Parse file naming: MOD11A1.A{YYYYDDD}.h{HH}v{VV}.061.{timestamp}.hdf
import re
from datetime import datetime, timedelta

tiles = set()
dates = []
for f in modis_hdf:
    fname = os.path.basename(f)
    m = re.match(r'MOD11A1\.A(\d{7})\.(h\d{2}v\d{2})\.', fname)
    if m:
        doy_str, tile = m.groups()
        tiles.add(tile)
        year = int(doy_str[:4])
        doy = int(doy_str[4:])
        date = datetime(year, 1, 1) + timedelta(days=doy - 1)
        dates.append(date)

dates = sorted(set(dates))
print(f"Tiles: {sorted(tiles)}")
print(f"Date range: {dates[0].strftime('%Y-%m-%d')} to {dates[-1].strftime('%Y-%m-%d')}")
print(f"Unique dates: {len(dates)}")
print(f"Files per tile: ~{len(modis_hdf) // len(tiles)}")

In [ ]:
# Examine one MODIS HDF file in detail
sample_hdf = modis_hdf[0]
print(f"Sample file: {os.path.basename(sample_hdf)}")
print(f"File size: {os.path.getsize(sample_hdf) / 1024:.0f} KB\n")

hdf = SD(sample_hdf, SDC.READ)
print("Available Scientific Data Sets (SDS):")
for name, info in hdf.datasets().items():
    dims, shape, dtype, nattrs = info
    print(f"  {name:30s} shape={shape} type={dtype}")

# Read and examine LST_Day_1km
lst_sds = hdf.select('LST_Day_1km')
lst_data = lst_sds.get()
attrs = lst_sds.attributes()
print(f"\nLST_Day_1km attributes:")
for k, v in attrs.items():
    print(f"  {k}: {v}")

scale = attrs.get('scale_factor', 0.02)
valid_range = attrs.get('valid_range', [7500, 65535])
fill_value = attrs.get('_FillValue', 0)

# Convert to Celsius
lst_valid = np.where((lst_data >= valid_range[0]) & (lst_data <= valid_range[1]),
                     lst_data * scale - 273.15, np.nan)

print(f"\nLST Day (Celsius): min={np.nanmin(lst_valid):.1f}, "
      f"max={np.nanmax(lst_valid):.1f}, "
      f"mean={np.nanmean(lst_valid):.1f}")
print(f"Valid pixel fraction: {np.sum(~np.isnan(lst_valid)) / lst_valid.size:.1%}")

hdf.end()

In [ ]:
def read_modis_lst(hdf_path, layer='LST_Day_1km'):
    """Read MODIS LST from HDF and convert to Celsius."""
    hdf = SD(hdf_path, SDC.READ)
    sds = hdf.select(layer)
    data = sds.get().astype(np.float64)
    attrs = sds.attributes()
    scale = attrs.get('scale_factor', 0.02)
    valid_range = attrs.get('valid_range', [7500, 65535])
    data = np.where((data >= valid_range[0]) & (data <= valid_range[1]),
                    data * scale - 273.15, np.nan)
    hdf.end()
    return data

# Plot LST for all 3 tiles on the same day
sample_date = os.path.basename(modis_hdf[0]).split('.')[1]  # e.g. A2022001
day_files = sorted([f for f in modis_hdf if sample_date in os.path.basename(f)])

n = len(day_files)
fig, axes = plt.subplots(1, n, figsize=(5 * n + 2, 5))
if n == 1:
    axes = [axes]

for ax, f in zip(axes, day_files):
    tile = re.search(r'(h\d{2}v\d{2})', os.path.basename(f)).group(1)
    lst = read_modis_lst(f)
    im = ax.imshow(lst, cmap='RdYlBu_r', vmin=-20, vmax=30)
    ax.set_title(f'{tile}\n{sample_date}')
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    valid_pct = np.sum(~np.isnan(lst)) / lst.size * 100
    ax.text(0.02, 0.02, f'{valid_pct:.0f}% valid', transform=ax.transAxes,
            fontsize=9, color='white', bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))

fig.suptitle(f'MODIS MOD11A1 Daytime LST — {sample_date}', fontsize=14)

# Add colorbar in its own dedicated axes on the right
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.65])  # [left, bottom, width, height]
fig.colorbar(im, cax=cbar_ax, label='LST Day (°C)')

# Give the suptitle room above the two-line ax titles
fig.subplots_adjust(top=0.82, right=0.90, left=0.06, wspace=0.35)

plt.show()

In [ ]:
# Temporal coverage: count valid pixels per day across all tiles
print("Scanning MODIS temporal coverage (this may take a minute)...")

from collections import defaultdict
daily_stats = defaultdict(lambda: {'valid': 0, 'total': 0, 'mean_lst': []})

# Sample every 10th date for speed
sampled_files = modis_hdf[::3]  # every 3rd file
for f in sampled_files:
    fname = os.path.basename(f)
    m = re.match(r'MOD11A1\.A(\d{7})\.(h\d{2}v\d{2})\.', fname)
    if not m:
        continue
    doy_str = m.group(1)
    year = int(doy_str[:4])
    doy = int(doy_str[4:])
    date = datetime(year, 1, 1) + timedelta(days=doy - 1)

    lst = read_modis_lst(f)
    valid = np.sum(~np.isnan(lst))
    daily_stats[date]['valid'] += valid
    daily_stats[date]['total'] += lst.size
    if valid > 0:
        daily_stats[date]['mean_lst'].append(np.nanmean(lst))

# Plot temporal coverage
sorted_dates = sorted(daily_stats.keys())
valid_fracs = [daily_stats[d]['valid'] / daily_stats[d]['total'] for d in sorted_dates]
mean_lsts = [np.mean(daily_stats[d]['mean_lst']) if daily_stats[d]['mean_lst'] else np.nan
             for d in sorted_dates]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.bar(sorted_dates, valid_fracs, width=2, color='steelblue', alpha=0.7)
ax1.set_ylabel('Valid pixel fraction')
ax1.set_title('MODIS LST — Temporal Coverage (sampled)')
ax1.set_ylim(0, 1)
ax1.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='50% threshold')
ax1.legend()

ax2.scatter(sorted_dates, mean_lsts, s=8, c='orangered', alpha=0.7)
ax2.set_ylabel('Mean LST (°C)')
ax2.set_xlabel('Date')
ax2.set_title('MODIS LST — Mean Temperature Over Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nScanned {len(sampled_files)} files across {len(sorted_dates)} dates")
print(f"Mean valid fraction: {np.mean(valid_fracs):.1%}")
print(f"LST range: {np.nanmin(mean_lsts):.1f}°C to {np.nanmax(mean_lsts):.1f}°C")

## 6. Sentinel-2 (10m, ~Bimonthly)

Sentinel-2 bands B03 (Green), B04 (Red), B08 (NIR) at 10m resolution. These are used to compute:
- **NDVI** = (B08 - B04) / (B08 + B04) — vegetation index
- **NDWI** = (B03 - B08) / (B03 + B08) — water/moisture index

Files are organized as individual band TIFs per date and tile (T13SDD, T13SED, T13TDE, T13TEE).

In [ ]:
# Inventory Sentinel-2 files
sen2_files = sorted(glob.glob(os.path.join(SEN2_DIR, '*.tif')))
print(f"Total Sentinel-2 TIF files: {len(sen2_files)}")

# Parse naming: S2A_MSIL2A_{date}T{time}_{orbit}_{tile}_{procdate}_{band}.tif
sen2_dates = set()
sen2_tiles = set()
sen2_bands = set()
for f in sen2_files:
    fname = os.path.basename(f)
    parts = fname.replace('.tif', '').split('_')
    date_str = parts[2][:8]  # YYYYMMDD
    tile = parts[4]          # e.g. T13SDD
    band = parts[-1]
    sen2_dates.add(date_str)
    sen2_tiles.add(tile)
    sen2_bands.add(band)

print(f"Tiles: {sorted(sen2_tiles)}")
print(f"Bands: {sorted(sen2_bands)}")
print(f"Unique dates: {len(sen2_dates)}")
print(f"Date range: {min(sen2_dates)} to {max(sen2_dates)}")
print(f"Scenes (date x tile): {len(sen2_files) // len(sen2_bands)}")

# Show first few dates
print(f"\nFirst 10 dates: {sorted(sen2_dates)[:10]}")

In [ ]:
# Load one Sentinel-2 scene (3 bands) and compute NDVI / NDWI
# Pick the first available date+tile combo
sample_sen2_date = sorted(sen2_dates)[0]
sample_sen2_tile = sorted(sen2_tiles)[0]

def find_sen2_band(date, tile, band, files):
    """Find the Sentinel-2 file for a given date, tile, band."""
    for f in files:
        fname = os.path.basename(f)
        parts = fname.replace('.tif', '').split('_')
        f_date = parts[2][:8]
        f_tile = parts[4]
        f_band = parts[-1]
        if f_date == date and f_tile == tile and f_band == band:
            return f
    return None

b03_file = find_sen2_band(sample_sen2_date, sample_sen2_tile, 'B03', sen2_files)
b04_file = find_sen2_band(sample_sen2_date, sample_sen2_tile, 'B04', sen2_files)
b08_file = find_sen2_band(sample_sen2_date, sample_sen2_tile, 'B08', sen2_files)

print(f"Sample scene: {sample_sen2_date}, tile {sample_sen2_tile}")

with rasterio.open(b03_file) as src:
    b03 = src.read(1).astype(np.float64)
    sen2_profile = src.profile.copy()
    print(f"  B03 (Green): shape={b03.shape}, CRS={src.crs}, res={src.res}")
    print(f"  Bounds: {src.bounds}")

with rasterio.open(b04_file) as src:
    b04 = src.read(1).astype(np.float64)

with rasterio.open(b08_file) as src:
    b08 = src.read(1).astype(np.float64)

# Compute indices (avoid division by zero)
ndvi = np.where((b08 + b04) > 0, (b08 - b04) / (b08 + b04), np.nan)
ndwi = np.where((b03 + b08) > 0, (b03 - b08) / (b03 + b08), np.nan)

print(f"\n  NDVI range: [{np.nanmin(ndvi):.3f}, {np.nanmax(ndvi):.3f}], mean={np.nanmean(ndvi):.3f}")
print(f"  NDWI range: [{np.nanmin(ndwi):.3f}, {np.nanmax(ndwi):.3f}], mean={np.nanmean(ndwi):.3f}")

In [ ]:
# Plot Sentinel-2 bands, NDVI, and NDWI
# Downsample to avoid Colab RAM crashes on large 10m rasters
MAX_DIM = 2048
step = max(1, max(b03.shape) // MAX_DIM)
b03_s, b04_s, b08_s = b03[::step, ::step], b04[::step, ::step], b08[::step, ::step]
ndvi_s, ndwi_s = ndvi[::step, ::step], ndwi[::step, ::step]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: raw bands
for ax, data, name, cmap in zip(axes[0],
                                  [b03_s, b04_s, b08_s],
                                  ['B03 (Green)', 'B04 (Red)', 'B08 (NIR)'],
                                  ['Greens', 'Reds', 'YlOrRd']):
    vmax = np.percentile(data[data > 0], 98) if np.any(data > 0) else 1
    im = ax.imshow(data, cmap=cmap, vmin=0, vmax=vmax)
    ax.set_title(name)
    plt.colorbar(im, ax=ax, shrink=0.8)

# Row 2: NDVI, NDWI, false color composite
im_ndvi = axes[1][0].imshow(ndvi_s, cmap='RdYlGn', vmin=-0.5, vmax=1)
axes[1][0].set_title('NDVI')
plt.colorbar(im_ndvi, ax=axes[1][0], shrink=0.8)

im_ndwi = axes[1][1].imshow(ndwi_s, cmap='RdYlBu', vmin=-0.5, vmax=0.5)
axes[1][1].set_title('NDWI')
plt.colorbar(im_ndwi, ax=axes[1][1], shrink=0.8)

# False color composite (NIR-Red-Green)
rgb = np.stack([b08_s, b04_s, b03_s], axis=-1)
p98 = np.percentile(rgb[rgb > 0], 98) if np.any(rgb > 0) else 1
rgb = np.clip(rgb / p98, 0, 1)
axes[1][2].imshow(rgb)
axes[1][2].set_title('False Color (NIR-R-G)')

for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(f'Sentinel-2 — {sample_sen2_date} — Tile {sample_sen2_tile}', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

del b03_s, b04_s, b08_s, ndvi_s, ndwi_s, rgb

## 7. Train / Val / Test Splits

The dataset includes `splits.csv` and `metadata.csv` with pre-defined temporal block splits:

| Split | Period | Purpose |
|-------|--------|---------|
| train | 2022 Jan–Sep | Training |
| val | 2022 Oct–Dec | Validation (seasonal shift) |
| test | 2023 | Temporal generalization |

Each row in `metadata.csv` is a MODIS date × tile sample matched to the nearest Sentinel-2, ASTER, and LULC.

In [ ]:
import pandas as pd

metadata = pd.read_csv(os.path.join(DATA_ROOT, 'metadata.csv'))
print(f"Total samples: {len(metadata)}\n")
print(metadata.groupby('split').agg(
    samples=('date', 'count'),
    date_min=('date', 'min'),
    date_max=('date', 'max'),
    tiles=('modis_tile', 'nunique'),
))
print(f"\n{metadata.head()}")